# Spot the Bot: September Data Understanding and Preparation

This notebook records our September data-preparation work: what we checked, what we found, and what we decided before moving into October modeling.

### Setup and Canonical Pipeline Imports

In [1]:
from pathlib import Path
import sys

import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent]:
    src_path = candidate / "src"
    if (src_path / "spotbot").exists():
        sys.path.insert(0, str(src_path))
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find src/spotbot. Run this notebook from the repository checkout.")

from spotbot.data import load_hc3_csv, load_hc3_from_huggingface
from spotbot.preprocessing import expand_answers, prepare_question_groups, remove_invalid_answers
from spotbot.splitting import (
    component_summary,
    make_group_split,
    verify_answer_text_isolation,
    verify_component_isolation,
    verify_split_isolation,
)

REPO_ROOT

**Finding:** We moved the reusable preprocessing steps into `src/spotbot` and import them here. That keeps this notebook focused on the investigation instead of maintaining a second copy of the pipeline.

### Load HC3 and Check Raw Schema

In [2]:
data_path = REPO_ROOT / "data" / "hc3_all.csv"
if data_path.exists():
    raw_hc3 = load_hc3_csv(data_path)
else:
    raw_hc3 = load_hc3_from_huggingface()

schema = pd.DataFrame({
    "column": raw_hc3.columns,
    "dtype": [str(raw_hc3[col].dtype) for col in raw_hc3.columns],
})
print(f"Raw question rows: {len(raw_hc3):,}")
display(schema)
display(raw_hc3.head(3))

Raw question rows: 24,322


            column    dtype
0         question   object
1    human_answers   object
2  chatgpt_answers   object
3            index  float64
4           source   object

                                            question  ...       source
0  Why is every book I hear about a " NY Times # ...  ...  reddit_eli5
1  If salt is so bad for cars , why do we use it ...  ...  reddit_eli5
2  Why do we still have SD TV channels when HD lo...  ...  reddit_eli5

[3 rows x 5 columns]

**Finding:** We confirmed that HC3 starts at the question level, and both answer columns contain lists. Because of that structure, we split questions before expanding answers.

### Original Size and Domain Distribution

In [3]:
raw_domain_counts = raw_hc3["source"].value_counts().rename_axis("source").reset_index(name="question_rows")
raw_domain_counts["pct"] = (raw_domain_counts["question_rows"] / len(raw_hc3) * 100).round(2)
display(raw_domain_counts)

        source  question_rows    pct
0  reddit_eli5          17112  70.36
1      finance           3933  16.17
2     medicine           1248   5.13
3      open_qa           1187   4.88
4    wiki_csai            842   3.46

**Finding:** We noticed that Reddit ELI5 makes up most of HC3. We decided to stratify by `source` so train, validation, and test keep similar domain mixes.

### Missing Values vs Empty Answer Lists

In [4]:
missing_values = raw_hc3[["question", "human_answers", "chatgpt_answers", "source"]].isna().sum().rename("missing_values")
empty_lists = pd.Series({
    "empty_human_answer_lists": int((raw_hc3["human_answers"].map(len) == 0).sum()),
    "empty_ai_answer_lists": int((raw_hc3["chatgpt_answers"].map(len) == 0).sum()),
    "rows_with_either_empty_list": int(((raw_hc3["human_answers"].map(len) == 0) | (raw_hc3["chatgpt_answers"].map(len) == 0)).sum()),
})
display(missing_values.to_frame())
display(empty_lists.to_frame("count"))

                 missing_values
question                      0
human_answers                 0
chatgpt_answers               0
source                        0

                             count
empty_human_answer_lists         0
empty_ai_answer_lists          455
rows_with_either_empty_list    455

**Finding:** We checked missing values separately from empty answer lists. Rows with an empty human or AI answer list cannot produce both labels, so we remove those rows before grouping duplicates.

### Normalize Text and Remove Empty-List Question Rows

In [5]:
question_groups, quality_report, duplicate_report = prepare_question_groups(raw_hc3)
quality = pd.DataFrame([quality_report.__dict__])
display(quality)

   original_question_rows  ...  blank_ai_answers_removed
0                   24322  ...                         0

[1 rows x 6 columns]

**Finding:** We normalize unicode and whitespace, then remove rows with empty answer lists. We do not remove long answers or balance classes during September data prep.

### Answer-Length Investigation and 2600-Word Cutoff Decision

In [6]:
raw_with_lengths = raw_hc3.copy()
raw_with_lengths = raw_with_lengths[(raw_with_lengths["human_answers"].map(len) > 0) & (raw_with_lengths["chatgpt_answers"].map(len) > 0)].copy()
raw_with_lengths["first_human_word_count"] = raw_with_lengths["human_answers"].map(lambda answers: len(str(answers[0]).split()))
raw_with_lengths["first_ai_word_count"] = raw_with_lengths["chatgpt_answers"].map(lambda answers: len(str(answers[0]).split()))
length_check = raw_with_lengths[["first_human_word_count", "first_ai_word_count"]].agg(["count", "mean", "median", "max"]).round(2)
long_human = int((raw_with_lengths["first_human_word_count"] > 2600).sum())
display(length_check)
print(f"Rows that would be removed by first-human-answer > 2600 cutoff: {long_human:,}")

Rows that would be removed by first-human-answer > 2600 cutoff: 2


        first_human_word_count  first_ai_word_count
count                 23867.00             23867.00
mean                    147.98               177.99
median                   94.00               177.00
max                    7904.00               639.00

**Finding:** We found length outliers, but the old 2600-word cutoff only looked at the first human answer. After comparing that behavior with the project goals, we decided to keep long answers and report length as a confound instead.

### Exact Duplicate Normalized-Question Investigation

In [7]:
duplicate_summary = pd.DataFrame([duplicate_report.__dict__])
display(duplicate_summary)

duplicate_sizes = (
    question_groups["original_row_count"]
    .value_counts()
    .rename_axis("raw_rows_per_question_group")
    .reset_index(name="question_groups")
    .sort_values("raw_rows_per_question_group")
)
display(duplicate_sizes)

   duplicated_question_groups  ...    chatgpt_answer_list_relationships
0                         528  ...  {'identical': 504, 'different': 24}

[1 rows x 5 columns]

   raw_rows_per_question_group  question_groups
0                            1            22811
1                            2              528

**Finding:** We found exact duplicate normalized questions. We decided to merge each duplicate question into one group, keep repeated identical answers once per label, and preserve distinct answers.

### Suspicious Upstream Duplicate-Question Quality Issue

In [8]:
gift_card_mask = question_groups["question"].str.contains("make money from gift cards", case=False, regex=False)
gift_card = question_groups.loc[gift_card_mask, ["question", "source", "original_row_count", "original_row_ids", "human_answers", "chatgpt_answers"]].head(1)
if not gift_card.empty:
    row = gift_card.iloc[0]
    display(pd.DataFrame([{
        "question_excerpt": row["question"][:160],
        "source": row["source"],
        "original_row_count": row["original_row_count"],
        "original_row_ids": row["original_row_ids"],
        "human_answer_count": len(row["human_answers"]),
        "ai_answer_count": len(row["chatgpt_answers"]),
    }]))
    print("Human answer excerpts:")
    for answer in row["human_answers"][:4]:
        print("-", answer[:180].replace("\n", " "))

Human answer excerpts:
- Also as you get older you will realise that Jerry is a dickhead , who spends his time annoying / pissing of Tom .
- I would say that most of the time the " bad " things or the epic fails always happen to Tom . He is usually frustrated or sad because he ca n't be as successful . It sort of comes 
- In a lot of episodes Tom is either just doing his job ( if he does n't stop Jerry from stealing food then he 'll probably get kicked out of the house , in some cases he has been ) 
- When you buy a gift card , you are giving the company an interest free loan . That in itself is good enough for them .


                                    question_excerpt  ... ai_answer_count
0  How do companies and stores make money from gi...  ...               2

[1 rows x 6 columns]

**Finding:** We found one duplicate gift-card question with unrelated Tom/Jerry human answers in the upstream data. We decided not to manually repair it because we cannot reliably reconstruct the original source row.

### Conservative Invalid/System Answer Removal

In [9]:
clean_groups, invalid_report = remove_invalid_answers(question_groups)
invalid_rows = []
for label, reason_counts in invalid_report.removed_by_label_reason.items():
    for reason, count in reason_counts.items():
        invalid_rows.append({"label": label, "reason": reason, "removed_answers": count})
invalid_summary = pd.DataFrame(invalid_rows).sort_values(["label", "reason"]) if invalid_rows else pd.DataFrame(columns=["label", "reason", "removed_answers"])
display(invalid_summary)
print(f"Total invalid/system answers removed: {invalid_report.total_removed:,}")

Total invalid/system answers removed: 186


   label                 reason  removed_answers
3     ai         api_rate_limit               51
1     ai           auth_failure               24
5     ai  chatgpt_security_page               26
2     ai  chatgpt_ui_transcript               55
4     ai          network_error               29
0  human         api_rate_limit                1

**Finding:** We remove only clear collection/system artifacts, such as rate-limit failures, auth failures, network errors, and ChatGPT UI page text. We keep short, generic, weak, or clarification-style answers so the rule stays conservative.

### Exact Answer Text Shared by Different Questions

In [10]:
pre_component_expanded, pre_component_blank_report = expand_answers(clean_groups)
answer_group_counts = (
    pre_component_expanded.groupby("text")["question_group_id"]
    .nunique()
    .rename("question_groups")
    .reset_index()
)
shared_answer_texts = answer_group_counts[answer_group_counts["question_groups"] > 1]
shared_summary = pd.DataFrame([{
    "shared_exact_answer_texts": len(shared_answer_texts),
    "max_question_groups_for_one_answer": int(shared_answer_texts["question_groups"].max()) if len(shared_answer_texts) else 0,
    "blank_ai_answers_found_before_expansion": pre_component_blank_report.blank_ai_answers_removed,
}])
display(shared_summary)

   shared_exact_answer_texts  ...  blank_ai_answers_found_before_expansion
0                       2958  ...                                       15

[1 rows x 3 columns]

**Finding:** We found exact answer text reused under different questions. Most remaining substantive cases are human answers on near-duplicate Reddit questions or synonym-like Wiki titles, so we keep linked question groups together when splitting.

### Connected-Component Leakage Solution

In [11]:
split_groups, split_summary = make_group_split(clean_groups, random_state=42)
components = component_summary(split_groups)
component_stats = pd.DataFrame([{
    "answer_linked_components": len(components),
    "multi_question_components": int((components["component_size"] > 1).sum()),
    "largest_component": int(components["component_size"].max()),
}])
component_size_counts = components["component_size"].value_counts().sort_index().rename_axis("component_size").reset_index(name="components")
display(component_stats)
display(component_size_counts)

   answer_linked_components  multi_question_components  largest_component
0                     22283                       1044                  4

   component_size  components
0               1       21239
1               2        1034
2               3           8
3               4           2

**Finding:** We split by answer-linked components. A one-question component behaves like a normal question group, while questions sharing exact answer text stay in the same split.

### Grouped 80/10/10 Split Before Expansion

In [12]:
split_summary_df = pd.DataFrame([split_summary.__dict__])
group_counts = split_groups["split"].value_counts().reindex(["train", "validation", "test"]).rename("question_groups")
component_counts = components.merge(split_groups[["component_id", "split"]].drop_duplicates(), on="component_id")["split"].value_counts().reindex(["train", "validation", "test"]).rename("components")
display(split_summary_df)
display(pd.concat([group_counts, component_counts], axis=1))

   train_groups  validation_groups  ...  validation_components  test_components
0         18671               2343  ...                   2228             2229

[1 rows x 6 columns]

            question_groups  components
split                                  
train                 18671       17826
validation             2343        2228
test                   2325        2229

**Finding:** We split before expanding answers and use `random_state=42`. The split stays close to 80/10/10 while keeping linked components together and preserving the source mix.

### Expand Answers and Preserve Labels

In [13]:
expanded, answer_report = expand_answers(split_groups)
expanded_counts = expanded["split"].value_counts().reindex(["train", "validation", "test"]).rename("expanded_examples")
blank_summary = pd.DataFrame([{
    "blank_human_answers_removed": answer_report.blank_human_answers_removed,
    "blank_ai_answers_removed": answer_report.blank_ai_answers_removed,
}])
display(expanded_counts.to_frame())
display(blank_summary)
display(expanded.head())

            expanded_examples
split                        
train                   65401
validation               8218
test                     8151

   blank_human_answers_removed  blank_ai_answers_removed
0                            0                        15

   split    question_group_id  ... label answer_index
0  train  qg_0001a99a9cfea00d  ...     0            0
1  train  qg_0001a99a9cfea00d  ...     0            1
2  train  qg_0001a99a9cfea00d  ...     0            2
3  train  qg_0001a99a9cfea00d  ...     1            0
4  train  qg_000299619139c47d  ...     0            0

[5 rows x 8 columns]

**Finding:** After splitting, we expand to one row per non-blank answer. We keep the question group, component, source, and split fields, with `0 = human` and `1 = AI`.

### Final Class Balance and Majority Baseline

In [14]:
label_counts = pd.crosstab(expanded["split"], expanded["label"]).rename(columns={0: "human", 1: "ai"}).reindex(["train", "validation", "test"])
label_pct = (label_counts.div(label_counts.sum(axis=1), axis=0) * 100).round(2)
majority_baseline = (label_counts.max(axis=1) / label_counts.sum(axis=1) * 100).round(2).rename("majority_baseline_accuracy_pct")
display(label_counts)
display(label_pct)
display(majority_baseline.to_frame())

label       human     ai
split                   
train       44473  20928
validation   5587   2631
test         5536   2615

label       human     ai
split                   
train       68.00  32.00
validation  67.98  32.02
test        67.92  32.08

            majority_baseline_accuracy_pct
split                                     
train                                68.00
validation                           67.98
test                                 67.92

**Finding:** We keep the natural class imbalance instead of deleting human answers. Class weighting or training-only resampling can be compared during October modeling, while validation and test stay naturally distributed.

### Final Domain Distribution

In [15]:
domain_group_counts = pd.crosstab(split_groups["source"], split_groups["split"]).reindex(columns=["train", "validation", "test"])
domain_group_pct = (domain_group_counts.div(split_groups["split"].value_counts().reindex(["train", "validation", "test"]), axis=1) * 100).round(2)
domain_answer_counts = pd.crosstab(expanded["source"], expanded["split"]).reindex(columns=["train", "validation", "test"])
display(domain_group_counts)
display(domain_group_pct)
display(domain_answer_counts)

split        train  validation  test
source                              
finance       3146         393   394
medicine       999         125   124
open_qa        949         119   119
reddit_eli5  12903        1622  1604
wiki_csai      674          84    84

split        train  validation   test
source                               
finance      16.85       16.77  16.95
medicine      5.35        5.34   5.33
open_qa       5.08        5.08   5.12
reddit_eli5  69.11       69.23  68.99
wiki_csai     3.61        3.59   3.61

split        train  validation  test
source                              
finance       6691         845   843
medicine      2038         251   258
open_qa       3759         471   474
reddit_eli5  51565        6483  6409
wiki_csai     1348         168   167

**Finding:** The source proportions are close across splits because we stratify components by source. Reddit ELI5 remains largest because it is largest in raw HC3.

### Final Answer-Length Distributions and Length Confound

In [16]:
eda = expanded.copy()
words = eda["text"].str.split()
eda["word_count"] = words.str.len()
eda["unique_word_ratio"] = words.apply(lambda tokens: len({token.lower() for token in tokens}) / len(tokens) if tokens else 0)
eda["punct_per_100_words"] = eda["text"].str.count(r"[^\w\s]") / eda["word_count"].clip(lower=1) * 100
length_stats = (
    eda.groupby(["split", "label"])[["word_count", "unique_word_ratio", "punct_per_100_words"]]
    .agg(["count", "mean", "median", "max"])
    .rename(index={0: "human", 1: "ai"}, level="label")
    .round(2)
)
display(length_stats)

                 word_count                 ... punct_per_100_words                
                      count    mean median  ...                mean median      max
split      label                            ...                                    
test       human       5536  132.99   83.0  ...               15.90  14.67   254.55
           ai          2615  174.87  175.0  ...               13.18  12.21    57.14
train      human      44473  134.74   83.0  ...               16.00  14.74   320.00
           ai         20928  174.12  175.0  ...               13.24  12.26  2400.00
validation human       5587  134.00   86.0  ...               16.06  14.68   163.64
           ai          2631  171.74  172.0  ...               13.08  12.20    48.31

[6 rows x 12 columns]

**Finding:** We see clear length and style differences between human and AI answers. We measure this directly instead of silently filtering long human examples.

### Leakage Validation

In [17]:
question_overlap = verify_split_isolation(split_groups)
answer_text_overlap = verify_answer_text_isolation(expanded)
component_overlap = verify_component_isolation(split_groups)
leakage = pd.DataFrame([
    {"check": "normalized_question_overlap", **{f"{a}/{b}": count for (a, b), count in question_overlap.items()}},
    {"check": "exact_answer_text_overlap", **{f"{a}/{b}": count for (a, b), count in answer_text_overlap.items()}},
    {"check": "component_overlap", **{f"{a}/{b}": count for (a, b), count in component_overlap.items()}},
])
display(leakage)

                         check  train/validation  train/test  validation/test
0  normalized_question_overlap                 0           0                0
1    exact_answer_text_overlap                 0           0                0
2            component_overlap                 0           0                0

**Finding:** We checked leakage after the final split. Train, validation, and test have zero normalized-question overlap, zero exact answer-text overlap, and zero component overlap.

### TF-IDF Feature Preparation Fits on Training Text Only

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

train_df = expanded[expanded["split"] == "train"].copy()
val_df = expanded[expanded["split"] == "validation"].copy()
test_df = expanded[expanded["split"] == "test"].copy()

tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["text"])
X_val = tfidf.transform(val_df["text"])
X_test = tfidf.transform(test_df["text"])

tfidf_shapes = pd.DataFrame([
    {"split": "train", "rows": X_train.shape[0], "features": X_train.shape[1], "operation": "fit_transform"},
    {"split": "validation", "rows": X_val.shape[0], "features": X_val.shape[1], "operation": "transform_only"},
    {"split": "test", "rows": X_test.shape[0], "features": X_test.shape[1], "operation": "transform_only"},
])
display(tfidf_shapes)

        split   rows  features       operation
0       train  65401     50000   fit_transform
1  validation   8218     50000  transform_only
2        test   8151     50000  transform_only

**Finding:** We fit TF-IDF on training text only. Validation and test are transformed with the fitted vectorizer, and this notebook does not train or select classifiers.

### Dataset Limitations to Carry Forward

In [19]:
limitations = pd.DataFrame([
    {"limitation": "HC3 was collected from early ChatGPT-era outputs, so model drift is likely."},
    {"limitation": "Answer length is a confound and must be reported during modeling."},
    {"limitation": "Some upstream rows contain collection/system artifacts; only clear deterministic artifacts are removed."},
    {"limitation": "One observed duplicate gift-card question contains likely misaligned human answers and is documented rather than manually repaired."},
    {"limitation": "Exact answer reuse across near-duplicate questions requires component-level splitting."},
])
display(limitations)

                                          limitation
0  HC3 was collected from early ChatGPT-era outpu...
1  Answer length is a confound and must be report...
2  Some upstream rows contain collection/system a...
3  One observed duplicate gift-card question cont...
4  Exact answer reuse across near-duplicate quest...

**Finding:** This model-ready dataset is our documented preparation of HC3, not the official raw HC3 answer count. We should carry these limitations into October modeling and the final report.